In [ ]:
import random
from itertools import combinations
from collections import defaultdict

docs = {}
for i in range(1, 5):
  with open(f"D{i}.txt") as f: # Open each file and read text
    docs[f"D{i}"] = f.read().strip()

In [ ]:
docs

{'D1': 'apple ceo tim cook is spending some time in canada this week and yesterday he attended a hockey game and visited the eaton centre apple store in toronto cook today stopped by the offices of canadian ecommerce platform shopify where he spoke to the financial post about augmented reality apps and the homepod on the topic of the homepod cook said that apples deep integration between hardware and software will help to differentiate the smart speaker from competing products like amazons alexa and the google home competition makes all of us better and i welcome it cook said but if you are both trying to license something and compete with your licensees this is a difficult model and it remains to be seen if it can be successful or not cook also said a quality very immersive audio experience was one thing missing from the smart speaker market which apple is aiming to fix music deserves that kind of quality as opposed to some kind of squeaky sound he said the homepod which at in the uni

In [ ]:
def char_kgrams(text, k):
  grams = set()  # set to avoid duplicates
  for i in range(len(text) - k + 1):
    grams.add(text[i:i+k])  # taking k characters
  return grams

def word_kgrams(text, k):
  words = text.split()  # split into words
  grams = set()
  for i in range(len(words) - k + 1):
    grams.add(tuple(words[i:i+k]))  # take k words
  return grams

# Building all required k-grams
grams_2_char = {d: char_kgrams(t, 2) for d, t in docs.items()}
grams_3_char = {d: char_kgrams(t, 3) for d, t in docs.items()}
grams_2_word = {d: word_kgrams(t, 2) for d, t in docs.items()}

# Jaccard similarity function
def jaccard(A, B):
  return len(A & B) / len(A | B)

# Compute all pair similarities (3 types × 6 pairs)
for name, grams in [("2-char", grams_2_char),
                    ("3-char", grams_3_char),
                    ("2-word", grams_2_word)]:
  print("\nType:", name)
  for d1, d2 in combinations(docs.keys(), 2):
    sim = jaccard(grams[d1], grams[d2])
    print(d1, d2, "=", round(sim, 3))


Type: 2-char
D1 D2 = 0.981
D1 D3 = 0.816
D1 D4 = 0.644
D2 D3 = 0.8
D2 D4 = 0.641
D3 D4 = 0.653

Type: 3-char
D1 D2 = 0.978
D1 D3 = 0.58
D1 D4 = 0.305
D2 D3 = 0.568
D2 D4 = 0.306
D3 D4 = 0.312

Type: 2-word
D1 D2 = 0.941
D1 D3 = 0.182
D1 D4 = 0.03
D2 D3 = 0.174
D2 D4 = 0.03
D3 D4 = 0.016


In [ ]:
g1 = grams_3_char["D1"]
g2 = grams_3_char["D2"]

for t in [20, 60, 150, 300, 600]:
  matches = 0  # count equal hash values

  for _ in range(t):
    # Create random hash function h(x) = (a*x + b) mod m
    a = random.randint(1, 10000)
    b = random.randint(0, 10000)

    # Compute # linear hash: (a * x + b) mod 10007 for each document
    min1 = min((a * abs(hash(x)) + b) % 10007 for x in g1)
    min2 = min((a * abs(hash(x)) + b) % 10007 for x in g2)
    if min1 == min2:
      matches += 1

  approx = matches / t
  print("t =", t, "Approx Jaccard =", round(approx, 3))

t = 20 Approx Jaccard = 1.0
t = 60 Approx Jaccard = 0.983
t = 150 Approx Jaccard = 0.973
t = 300 Approx Jaccard = 0.967
t = 600 Approx Jaccard = 0.98


In [ ]:
t = 160
r = 5          # number of rows
b = t // r     # number of bands

# Build signatures for all documents
hash_funcs = []
for _ in range(t):
  a = random.randint(1, 10000)
  b_rand = random.randint(0, 10000)
  hash_funcs.append((a, b_rand))

signatures = {}

# Step 2: Use the hash functions for each document
for d, grams in grams_3_char.items():
  sig = []
  for a, b_rand in hash_funcs:
    # Compute min hash value function
    m = min((a * abs(hash(x)) + b_rand) % 10007 for x in grams)
    sig.append(m)
  signatures[d] = sig

# LSH bucketing
buckets = defaultdict(list)
for d, sig in signatures.items():
  for band in range(b):
    start = band * r
    part = tuple(sig[start:start+r])  # part of signature
    buckets[(band, part)].append(d)

# Candidate pairs
candidates = set()
for group in buckets.values():
  if len(group) > 1:
    for pair in combinations(group, 2):
      candidates.add(pair)

print("Candidate pairs:", candidates)

Candidate pairs: {('D1', 'D2')}


In [ ]:
r = 5          # number of bands
b = 160 // r   # number of bands

# Function for S-curve probability
def lsh_probability(s, r, b):
  return 1 - (1 - s**r) ** b
# Using 3-gram similarities
grams = grams_3_char
for d1, d2 in combinations(docs.keys(), 2):
  s = jaccard(grams[d1], grams[d2])   # true similarity
  prob = lsh_probability(s, r, b)
  print(d1, d2,
        "Similarity =", round(s, 3),
        "Probability =", round(prob, 3))

D1 D2 Similarity = 0.978 Probability = 1.0
D1 D3 Similarity = 0.58 Probability = 0.887
D1 D4 Similarity = 0.305 Probability = 0.081
D2 D3 Similarity = 0.568 Probability = 0.858
D2 D4 Similarity = 0.306 Probability = 0.082
D3 D4 Similarity = 0.312 Probability = 0.091


In [ ]:
!unzip ml-100k.zip

Archive:  ml-100k.zip
replace ml-100k/allbut.pl? [y]es, [n]o, [A]ll, [N]one, [r]ename: N


In [ ]:
# Reading MovieLens data
user_movies = defaultdict(set)
with open("ml-100k/u.data") as f:
  for line in f:
    user, movie, rating, time = line.split('\t')
    user_movies[int(user)].add(int(movie))
users = list(user_movies.keys())

# similar pairs ≥ 0.5
true_pairs = set()
for u1, u2 in combinations(users, 2):
  sim = jaccard(user_movies[u1], user_movies[u2])
  if sim >= 0.5:
    true_pairs.add((u1, u2))
print("True pairs ≥ 0.5:", len(true_pairs))

# MinHash experiments
for t in [50, 100, 200]:
  total_fp = 0
  total_fn = 0

  # Run 5 times and average
  for run in range(5):
    # Create hash functions
    hash_funcs = [(random.randint(1, 10000),
                  random.randint(0, 10000))
                  for _ in range(t)]
    # Building signatures
    signatures = {}
    for u in users:
      sig = []

      for a, b in hash_funcs:
        m = min((a * m_id + b) % 10007
                for m_id in user_movies[u])
        sig.append(m)
      signatures[u] = sig
    approx_pairs = set()

    for u1, u2 in combinations(users, 2):
      matches = 0
      for i in range(t):
        if signatures[u1][i] == signatures[u2][i]:
          matches += 1

      if matches / t >= 0.5:
        approx_pairs.add((u1, u2))

    fp = len(approx_pairs - true_pairs)
    fn = len(true_pairs - approx_pairs)

    total_fp += fp
    total_fn += fn

  print("t =", t,
        "Avg FP =", total_fp / 5,
        "Avg FN =", total_fn / 5)

True pairs ≥ 0.5: 10
t = 50 Avg FP = 154.0 Avg FN = 2.8
t = 100 Avg FP = 38.8 Avg FN = 2.0
t = 200 Avg FP = 6.8 Avg FN = 2.6


In [ ]:
# true pairs for threshold 0.6
true_pairs_06 = set()
for u1, u2 in combinations(users, 2):
  if jaccard(user_movies[u1], user_movies[u2]) >= 0.6:
    true_pairs_06.add((u1, u2))

# configurations
configs = [
    (50, 5, 10),
    (100, 5, 20),
    (200, 5, 40),
    (200, 10, 20)
]

for threshold in [0.6, 0.8]:
  print("\nThreshold =", threshold)

  # true pairs for this threshold
  true_pairs = set()
  for u1, u2 in combinations(users, 2):
    if jaccard(user_movies[u1], user_movies[u2]) >= threshold:
      true_pairs.add((u1, u2))

  for t, r, b in configs:
    total_fp = 0
    total_fn = 0
    for run in range(5):
      # Build signatures
      hash_funcs = [(random.randint(1, 10000),
                      random.randint(0, 10000))
                    for _ in range(t)]
      signatures = {}
      for u in users:
        sig = []
        for a, bb in hash_funcs:
          m = min((a*m_id + bb) % 10007
                  for m_id in user_movies[u])
          sig.append(m)
        signatures[u] = sig

      # LSH
      buckets = defaultdict(list)
      for u, sig in signatures.items():
        for band in range(b):
          start = band * r
          part = tuple(sig[start:start+r])
          buckets[(band, part)].append(u)
      candidates = set()

      for group in buckets.values():
        if len(group) > 1:
            for pair in combinations(group, 2):
              candidates.add(pair)
      approx_pairs = set()

      for u1, u2 in candidates:
        if jaccard(user_movies[u1], user_movies[u2]) >= threshold:
          approx_pairs.add((u1, u2))

      total_fp += len(approx_pairs - true_pairs)
      total_fn += len(true_pairs - approx_pairs)

    print("t =", t, "r =", r, "b =", b,
          "Avg FP =", total_fp/5,
          "Avg FN =", total_fn/5)


Threshold = 0.6
t = 50 r = 5 b = 10 Avg FP = 0.0 Avg FN = 0.2
t = 100 r = 5 b = 20 Avg FP = 0.0 Avg FN = 0.4
t = 200 r = 5 b = 40 Avg FP = 0.0 Avg FN = 0.0
t = 200 r = 10 b = 20 Avg FP = 0.0 Avg FN = 1.4

Threshold = 0.8
t = 50 r = 5 b = 10 Avg FP = 0.0 Avg FN = 0.0
t = 100 r = 5 b = 20 Avg FP = 0.0 Avg FN = 0.0
t = 200 r = 5 b = 40 Avg FP = 0.0 Avg FN = 0.0
t = 200 r = 10 b = 20 Avg FP = 0.0 Avg FN = 0.0
